# RAG Pipeline (Retrieval-Augmented Generation)

## Latar Belakang

Tahap ini merupakan puncak dari keseluruhan sistem, yang merangkai seluruh komponen yang telah dibangun sebelumnya menjadi satu sistem tanya-jawab yang utuh. Pada tahap-tahap terdahulu telah dihasilkan basis pengetahuan dalam bentuk vector database, model klasifikasi kategori, dan model analisis sentimen. Komponen-komponen tersebut kini disatukan melalui pendekatan Retrieval-Augmented Generation (RAG), sehingga pengguna dapat mengajukan pertanyaan dan memperoleh jawaban berbahasa alami yang berlandaskan artikel berita nyata. Sebelum membahas implementasinya, bagian ini terlebih dahulu menguraikan konsep RAG beserta komponen penyusunnya sebagai dasar pemahaman.

## Apa itu RAG?

Retrieval-Augmented Generation (RAG) adalah pendekatan yang menggabungkan pencarian informasi dari sumber pengetahuan eksternal dengan kemampuan generasi teks dari Large Language Model (LLM). Alih-alih menjawab semata-mata dari pengetahuan internal model, sistem RAG terlebih dahulu mencari potongan teks yang relevan dari basis pengetahuan, lalu menyerahkannya kepada LLM sebagai konteks untuk menyusun jawaban. Dengan demikian, jawaban yang dihasilkan berlandaskan pada data nyata yang dapat dikendalikan dan ditelusuri sumbernya. Istilah RAG sendiri tersusun dari tiga komponen, yaitu Retrieval, Augmented, dan Generation.

## Retrieval (Pengambilan)

Retrieval adalah tahap pencarian potongan teks yang paling relevan dengan pertanyaan pengguna. Pertanyaan diubah menjadi vektor menggunakan model embedding yang sama dengan yang dipakai saat membangun basis pengetahuan, kemudian vector database mencari chunk dengan vektor terdekat berdasarkan kedekatan makna. Pada tahap ini, penyaringan metadata (seperti kategori dan sentimen) juga dapat diterapkan agar pencarian lebih terarah. Hasil retrieval inilah yang menjadi bahan konteks bagi tahap selanjutnya.

## Augmented (Penambahan Konteks)

Augmented merujuk pada proses memperkaya masukan LLM dengan konteks hasil retrieval. Potongan teks yang ditemukan tidak dibiarkan terpisah, melainkan disisipkan ke dalam prompt bersama pertanyaan pengguna. Dengan kata lain, kemampuan LLM "ditambah" (augmented) dengan pengetahuan eksternal yang relevan, sehingga model tidak hanya mengandalkan apa yang telah dipelajarinya saat pelatihan, tetapi juga informasi spesifik dan terkini dari basis pengetahuan. Tahap ini menjadi jembatan antara pencarian dan generasi jawaban.

## Generation (Penyusunan Jawaban)

Generation adalah tahap ketika LLM menghasilkan jawaban berbahasa alami berdasarkan prompt yang telah diperkaya konteks. Model membaca konteks yang diberikan, lalu menyusun jawaban yang ringkas dan koheren sesuai pertanyaan, sekaligus dapat merujuk sumber yang dipakai. Karena jawaban dibangun di atas konteks yang disediakan, hasilnya cenderung lebih akurat dan relevan dibandingkan jawaban yang dihasilkan murni dari pengetahuan internal model.

## Grounding dan Pencegahan Halusinasi

Salah satu konsep kunci dalam RAG adalah grounding, yaitu memastikan jawaban benar-benar bersandar pada konteks yang diberikan. Tanpa grounding, LLM berpotensi mengalami halusinasi, yaitu menghasilkan informasi yang terdengar meyakinkan namun tidak berdasar pada data. Grounding dicapai melalui instruksi prompt yang tegas, yang mengharuskan model menjawab hanya dari konteks dan secara jujur menyatakan apabila informasi tidak ditemukan, alih-alih mengarang. Permintaan untuk menyebutkan nomor sumber juga memperkuat grounding karena membuat jawaban dapat ditelusuri.

## Mengapa Menggunakan RAG?

Pendekatan RAG mengatasi sejumlah keterbatasan LLM yang digunakan secara mandiri. LLM murni memiliki batas pengetahuan (knowledge cutoff), berisiko berhalusinasi, dan tidak dapat menunjukkan sumber. Dengan RAG, jawaban dapat bersumber dari basis pengetahuan yang terkini, terkontrol, dan dapat ditelusuri, tanpa perlu melatih ulang model setiap kali datanya berubah. Pada penelitian ini, pendekatan RAG juga menjadi wadah integrasi tiga komponen, yaitu klasifikasi untuk pengarahan pencarian (routing), analisis sentimen untuk penyaringan dan agregasi, serta retrieval dan generation sebagai inti tanya-jawab.

In [1]:
!pip install -q google-genai groq sentence-transformers chromadb transformers torch

## Konfigurasi Provider dan Kunci API LLM

Sistem dirancang fleksibel terhadap penyedia LLM, mendukung dua layanan gratis yang dapat dipilih melalui variabel `LLM_PROVIDER`, yaitu Gemini dan Groq. Gemini (`gemini-2.5-flash`) ditetapkan sebagai default karena memiliki batas token per menit yang jauh lebih besar dan jendela konteks yang luas, sehingga lebih sesuai untuk RAG yang prompt-nya cenderung padat token akibat penyertaan konteks. Groq (`llama-3.3-70b-versatile`) disediakan sebagai alternatif yang sangat cepat. Penggantian provider cukup dilakukan dengan mengubah satu variabel konfigurasi.

Kunci API dimasukkan secara aman melalui `getpass`, sehingga tidak tersimpan di dalam notebook dan tidak ikut terbawa apabila notebook dibagikan atau diunggah ke repositori.

In [2]:
# === PILIH PROVIDER ===
LLM_PROVIDER = "gemini"        
GEMINI_MODEL = "gemini-2.5-flash"
GROQ_MODEL   = "llama-3.3-70b-versatile"

import os, getpass
if LLM_PROVIDER == "gemini":
    if not os.environ.get("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste Gemini API key: ")
elif LLM_PROVIDER == "groq":
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Paste Groq API key: ")
print(f"Provider: {LLM_PROVIDER}")

Paste Gemini API key:  ········


Provider: gemini


## Pemuatan dan Penghubungan Komponen

In [3]:
from pathlib import Path

def find_project_root():
    cwd = Path.cwd().resolve()
    if (cwd / "data").exists() and (cwd / "notebooks").exists():
        return cwd
    for parent in cwd.parents:
        if (parent / "data").exists() and (parent / "notebooks").exists():
            return parent
    return cwd

PROJECT_ROOT = find_project_root()
CHROMA_DIR = PROJECT_ROOT / "chroma_db"
CLASSIFIER_PATH = PROJECT_ROOT / "models" / "indobert_classifier"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
COLLECTION_NAME = "news_articles"

import torch
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ChromaDB
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(COLLECTION_NAME)
print(f"ChromaDB connected. Total chunk: {collection.count()}")

# Embedding model (SAMA dengan Day 4)
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print("Embedding model loaded.")

# Classifier kategori (Day 3)
device = 0 if torch.cuda.is_available() else -1
clf = pipeline("text-classification", model=str(CLASSIFIER_PATH),
               tokenizer=str(CLASSIFIER_PATH), device=device, truncation=True, max_length=256)
print("Classifier loaded.")

ChromaDB connected. Total chunk: 4448
Embedding model loaded.


Device set to use cuda:0


Classifier loaded.


Tahap ini menghubungkan kembali seluruh komponen yang telah dibangun pada tahap-tahap sebelumnya menjadi satu kesatuan yang siap dipakai, tanpa perlu melatih ulang model maupun membuat embedding dari awal. Pendekatan ini mencerminkan sifat modular sistem: setiap komponen dibangun dan disimpan secara terpisah, lalu dimuat saat dibutuhkan, sehingga proses menjadi efisien dan mudah dipelihara. Ada tiga komponen yang dimuat pada tahap ini.

Pertama, vector database (ChromaDB) dihubungkan dari penyimpanan permanen yang berisi 4.448 chunk beserta metadatanya. Komponen ini berperan sebagai basis pengetahuan tempat pencarian dilakukan, dan karena bersifat persisten, ia langsung dapat digunakan tanpa proses pembangunan ulang.

Kedua, model embedding dimuat. Penting untuk ditekankan bahwa model yang digunakan harus sama persis dengan model yang dipakai saat mengindeks chunk pada tahap pembangunan vector database. Konsistensi ini bersifat krusial karena pencarian semantik membandingkan vektor query dengan vektor chunk; apabila keduanya dihasilkan oleh model berbeda, keduanya berada pada ruang vektor yang tidak sebanding sehingga pencarian menjadi tidak valid.

Ketiga, model klasifikasi kategori dimuat melalui pipeline klasifikasi. Komponen ini berperan mengklasifikasikan pertanyaan pengguna pada saat sistem berjalan, yang hasilnya digunakan untuk mengarahkan (routing) dan menyaring pencarian ke kategori yang relevan. Dengan ketiga komponen ini termuat, sistem siap menjalankan keseluruhan alur RAG.

## Fungsi Pemanggil LLM (Abstraksi Provider)

Pemanggilan LLM dibungkus dalam satu fungsi `generate()` yang secara otomatis menyesuaikan dengan provider yang dipilih, sehingga sisa pipeline tidak perlu mengetahui detail teknis masing-masing layanan. Pendekatan abstraksi ini membuat sistem mudah berpindah antara Gemini dan Groq tanpa mengubah kode inti.

Fungsi ini menggunakan temperature rendah (0,3). Temperature mengatur tingkat keacakan keluaran model: nilai yang rendah membuat jawaban lebih deterministik dan setia pada konteks yang diberikan, sedangkan nilai tinggi mendorong variasi yang lebih kreatif. Untuk RAG, nilai rendah dipilih karena tujuannya adalah jawaban yang faithful terhadap konteks, bukan jawaban yang kreatif namun berisiko menyimpang. Sebagai pemeriksaan awal, fungsi diuji dengan satu permintaan sederhana untuk memastikan koneksi API berfungsi sebelum pipeline penuh dijalankan.

In [4]:
gemini_client = None
groq_client = None
if LLM_PROVIDER == "gemini":
    from google import genai
    gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
elif LLM_PROVIDER == "groq":
    from groq import Groq
    groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def generate(system_prompt, user_prompt, temperature=0.3):
    """Panggil LLM sesuai provider. Return string jawaban."""
    if LLM_PROVIDER == "gemini":
        from google.genai import types
        resp = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt, temperature=temperature),
        )
        return resp.text
    else:
        resp = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user", "content": user_prompt}],
            temperature=temperature,
        )
        return resp.choices[0].message.content

# Sanity check: pastiin API jalan
print(generate("Kamu asisten singkat.", "Balas dengan satu kalimat: apa itu RAG?"))

RAG (Retrieval-Augmented Generation) adalah teknik yang menggabungkan pengambilan informasi dari sumber eksternal dengan model bahasa besar untuk menghasilkan respons yang lebih akurat dan relevan.


RAG (Retrieval-Augmented Generation) adalah teknik yang menggabungkan pengambilan informasi dari sumber eksternal dengan model bahasa besar untuk menghasilkan respons yang lebih akurat dan relevan.

## Fungsi-Fungsi Komponen Pipeline

Sebelum dirangkai menjadi satu alur utuh, pipeline dibangun dari fungsi-fungsi kecil yang masing-masing mewakili satu langkah pada saat sistem berjalan. Fungsi `classify_query` memprediksi kategori dari pertanyaan pengguna beserta skor keyakinannya, yang digunakan untuk pengarahan dan penyaringan. Fungsi `retrieve` mengubah pertanyaan menjadi vektor, mencari chunk terdekat di ChromaDB (dengan filter metadata bila ada), lalu melakukan deduplikasi per artikel. Fungsi `aggregate_sentiment` menghitung distribusi sentimen dari chunk yang diambil, sedangkan `build_context` menyusun chunk menjadi teks konteks bernomor agar jawaban dapat menyertakan sitasi sumber.

Konsep deduplikasi pada fungsi `retrieve` penting untuk diperhatikan. Karena artikel panjang terpecah menjadi beberapa chunk, satu artikel yang sama berpotensi muncul berkali-kali pada hasil pencarian. Untuk mengatasinya, sistem mengambil lebih banyak kandidat terlebih dahulu, lalu menyaring agar hanya chunk terbaik (berjarak terkecil) dari tiap artikel yang dipertahankan. Dengan demikian, konteks yang dikirim ke LLM menjadi lebih efisien dan daftar sumber tidak berulang, sebagaimana telah diidentifikasi sebagai kebutuhan pada tahap pembangunan vector database.

In [5]:
from collections import Counter

def classify_query(query):
    r = clf(query)[0]
    return r["label"], r["score"]

def retrieve(query, k=5, where=None):
    """Retrieve top-k chunk, dedupe per article_id (ambil chunk jarak terkecil per artikel)."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).tolist()
    pool = k * 4  # ambil lebih banyak dulu, baru dedupe
    res = collection.query(query_embeddings=q_emb, n_results=pool, where=where)
    seen, out = set(), []
    for i in range(len(res["ids"][0])):
        m = res["metadatas"][0][i]
        if m["article_id"] in seen:
            continue
        seen.add(m["article_id"])
        out.append({"text": res["documents"][0][i],
                    "distance": res["distances"][0][i], **m})
        if len(out) >= k:
            break
    return out

def aggregate_sentiment(chunks):
    return dict(Counter(ch["sentiment"] for ch in chunks))

def build_context(chunks):
    parts = []
    for i, ch in enumerate(chunks, 1):
        parts.append(f"[Sumber {i}] ({ch['kategori']}/{ch['sentiment']}) {ch['judul']}\n{ch['text']}")
    return "\n\n".join(parts)

print("Fungsi komponen siap.")

Fungsi komponen siap.


## Prompt Grounding

Inti dari RAG bukan sekadar menyediakan konteks, melainkan mengarahkan LLM agar menjawab hanya berdasarkan konteks tersebut. Tanpa instruksi yang tegas, model berpotensi mengarang (hallucination) dari pengetahuan internalnya. Oleh karena itu, disusun sebuah system prompt yang mengoperasionalkan prinsip grounding.

Prompt ini memuat beberapa instruksi kunci sebagai praktik terbaik RAG, yaitu menegaskan agar model menjawab hanya dari konteks yang diberikan, secara jujur menyatakan ketika informasi tidak ditemukan alih-alih mengarang, menyertakan nomor sumber yang dipakai agar jawaban dapat ditelusuri, serta menjaga gaya jawaban tetap ringkas dan berbahasa Indonesia. Kombinasi instruksi inilah yang menjadi penjaga utama agar jawaban sistem tetap akurat dan tepercaya.

In [6]:
SYSTEM_PROMPT = (
    "Kamu asisten berita berbahasa Indonesia. Jawab pertanyaan HANYA berdasarkan konteks artikel "
    "yang diberikan di bawah. Jika informasinya tidak ada di konteks, katakan dengan jujur bahwa "
    "kamu tidak menemukannya dalam artikel yang tersedia - JANGAN mengarang. Jawab ringkas, jelas, "
    "dalam bahasa Indonesia, dan sebutkan nomor sumber yang kamu pakai (misal [1], [2])."
)
print(SYSTEM_PROMPT)

Kamu asisten berita berbahasa Indonesia. Jawab pertanyaan HANYA berdasarkan konteks artikel yang diberikan di bawah. Jika informasinya tidak ada di konteks, katakan dengan jujur bahwa kamu tidak menemukannya dalam artikel yang tersedia - JANGAN mengarang. Jawab ringkas, jelas, dalam bahasa Indonesia, dan sebutkan nomor sumber yang kamu pakai (misal [1], [2]).


## Fungsi Utama: `rag_answer()`

Seluruh langkah dirangkai menjadi satu fungsi utama `rag_answer()` yang menjalankan alur lengkap, mulai dari klasifikasi pertanyaan, pembangunan filter, retrieval dengan deduplikasi, agregasi sentimen, penyusunan prompt, hingga generation jawaban. Fungsi ini mengembalikan jawaban beserta daftar sumber, kategori yang terdeteksi, filter yang dipakai, dan ringkasan sentimen.

Beberapa keputusan desain diterapkan agar pipeline tangguh. Pada bagian routing, pencarian baru difilter ke kategori hasil prediksi apabila keyakinan classifier memadai (di atas ambang 0,5); ketika keyakinan rendah, sistem tidak memaksakan filter sehingga pencarian tetap luas dan relevan. Untuk menghindari hasil yang kosong, diterapkan mekanisme fallback: apabila penyaringan menghasilkan chunk yang terlalu sedikit, pencarian diulang tanpa filter. Penyaringan berdasarkan sentimen bersifat opsional dan hanya ditambahkan ketika pengguna secara eksplisit memintanya. Apabila tidak ada artikel relevan sama sekali, fungsi mengembalikan pesan yang jujur alih-alih memaksakan jawaban.

In [7]:
def rag_answer(query, k=5, route_by_category=True, sentiment=None, conf_threshold=0.5):
    # 1. classify query
    cat, cat_conf = classify_query(query)

    # 2. bangun filter metadata
    conds, used = [], {}
    if route_by_category and cat_conf >= conf_threshold:
        conds.append({"kategori": cat}); used["kategori"] = cat
    if sentiment:
        conds.append({"sentiment": sentiment}); used["sentiment"] = sentiment
    where = conds[0] if len(conds) == 1 else ({"$and": conds} if conds else None)

    # 3. retrieve + dedupe (fallback kalau hasil filter sedikit)
    chunks = retrieve(query, k=k, where=where)
    if route_by_category and where is not None and len(chunks) < k:
        chunks = retrieve(query, k=k, where=None)
        used["fallback"] = "tanpa filter (hasil filter sedikit)"

    if not chunks:
        return {"answer": "Maaf, tidak ada artikel relevan yang ditemukan.",
                "sources": [], "kategori_query": cat, "kategori_conf": cat_conf,
                "filter": used, "sentimen": {}}

    # 4. agregasi sentimen
    sent_summary = aggregate_sentiment(chunks)

    # 5. susun prompt + 6. generate
    context = build_context(chunks)
    user_prompt = f"Konteks artikel:\n\n{context}\n\nPertanyaan: {query}\n\nJawaban:"
    answer = generate(SYSTEM_PROMPT, user_prompt)

    return {"answer": answer,
            "sources": [{"judul": ch["judul"], "kategori": ch["kategori"],
                         "sentiment": ch["sentiment"], "url": ch["url"],
                         "distance": round(ch["distance"], 3)} for ch in chunks],
            "kategori_query": cat, "kategori_conf": round(cat_conf, 2),
            "filter": used, "sentimen": sent_summary}

print("rag_answer() siap.")

rag_answer() siap.


## Tes Pipeline

Pipeline diuji dengan beberapa tipe pertanyaan untuk memastikan seluruh komponen bekerja secara terpadu. Sebuah fungsi bantu `tampilkan()` dibuat untuk menyajikan hasil secara ringkas, mencakup kategori terdeteksi, filter yang dipakai, ringkasan sentimen, jawaban, dan daftar sumber.


In [8]:
def tampilkan(query, **kwargs):
    print("=" * 78)
    print(f"PERTANYAAN: {query}")
    if kwargs:
        print(f"(opsi: {kwargs})")
    print("=" * 78)
    out = rag_answer(query, **kwargs)
    print(f"\nKategori query terdeteksi : {out['kategori_query']} ({out['kategori_conf']})")
    print(f"Filter dipakai            : {out['filter']}")
    print(f"Ringkasan sentimen sumber : {out['sentimen']}")
    print(f"\nJAWABAN:\n{out['answer']}")
    print(f"\nSUMBER:")
    for i, s in enumerate(out["sources"], 1):
        print(f"  [{i}] ({s['kategori']}/{s['sentiment']}, dist={s['distance']}) {s['judul'][:60]}")
        print(f"      {s['url']}")
    print()

tampilkan("Apa yang terjadi dengan IHSG belakangan ini?")

PERTANYAAN: Apa yang terjadi dengan IHSG belakangan ini?

Kategori query terdeteksi : finance (0.91)
Filter dipakai            : {'kategori': 'finance'}
Ringkasan sentimen sumber : {'netral': 5}

JAWABAN:
Belakangan ini, Indeks Harga Saham Gabungan (IHSG) cenderung melemah. Pada Jumat (29/5/2026), IHSG ditutup melemah ke level 6.127 [1]. Sebelumnya, pada Selasa (26/5/2026), IHSG juga ditutup turun 1,23% ke level 6.130,19 di tengah derasnya arus keluar dana asing [3].

Meskipun sempat menguat pada Senin (25/5/2026) ke level 6.206,34 [2], IHSG kembali melemah pada Jumat (22/5/2026), bahkan sempat menukik turun hingga rentang terendah 5.966 dan menyentuh level 5.000-an [4]. Pada Kamis (21/5/2026), IHSG ditutup anjlok 3,54% ke level 6.094 [5].

Secara keseluruhan, IHSG juga menunjukkan pelemahan secara bulanan, tiga bulanan, dan sepanjang tahun 2026 [4, 5].

SUMBER:
  [1] (finance/netral, dist=0.42) IHSG Tiba-tiba Terjun Bebas, Saham Big Bank Rontok!
      https://finance.detik.com/bursa-d

Pada pengujian pertama, pertanyaan "Apa yang terjadi dengan IHSG belakangan ini?" diklasifikasikan sebagai finance dengan keyakinan tinggi (0,91), sehingga pencarian difilter ke kategori tersebut. Sistem menghasilkan jawaban yang menyintesis informasi dari lima sumber, lengkap dengan angka dan tanggal spesifik (misalnya level penutupan IHSG dan persentase pelemahan) serta sitasi nomor sumber pada setiap pernyataan. Hasil ini menunjukkan alur penuh berjalan sebagaimana mestinya: klasifikasi mengarahkan pencarian, retrieval menyediakan konteks yang relevan, dan LLM menyusun jawaban yang akurat dan dapat ditelusuri sumbernya.

In [9]:
tampilkan("Berita teknologi atau gadget terbaru apa saja?")

PERTANYAAN: Berita teknologi atau gadget terbaru apa saja?

Kategori query terdeteksi : inet (0.85)
Filter dipakai            : {'kategori': 'inet'}
Ringkasan sentimen sumber : {'positif': 2, 'netral': 3}

JAWABAN:
Berikut adalah beberapa berita teknologi atau gadget terbaru yang disebutkan dalam artikel:

*   Erafone meluncurkan kampanye "Erafone Lebih Dekat" untuk menghadirkan pengalaman belanja gadget yang lebih praktis dan mudah dijangkau, serta menyediakan berbagai pilihan *smartphone* dan perangkat elektronik [1].
*   Telkomsel mengungkapkan strategi baru untuk menjadi perusahaan digital berbasis AI, dengan memperluas penggunaan AI untuk operasional jaringan (Autonomous Network), layanan pelanggan (Virtual Assistant Veronika), dan menghadirkan fitur pembelajaran berbasis AI bernama Sacred Octagon di aplikasi MyTelkomsel. Telkomsel juga mempercepat ekspansi jaringan Hyper 5G [2].
*   Tren micro drama atau "Drachin Pendek" dengan format mirip TikTok dan Reels diprediksi akan menjad

Pengujian kedua menggunakan pertanyaan "Berita teknologi atau gadget terbaru apa saja?" yang diklasifikasikan sebagai inet dengan keyakinan 0,85, sehingga pencarian difilter ke kategori teknologi. Jawaban yang dihasilkan merangkum beberapa berita teknologi terkini dari sumber yang ditemukan (antara lain kampanye Erafone, strategi AI Telkomsel, dan tren micro drama), masing-masing disertai sitasi sumber. Hasil ini memperlihatkan kemampuan sistem menjawab pertanyaan yang bersifat ringkasan atas suatu kategori, bukan hanya pertanyaan dengan satu fokus spesifik.

In [10]:
# Pakai filter sentimen: minta yang positif
tampilkan("Ada kabar olahraga Indonesia yang membanggakan?", sentiment="positif")

PERTANYAAN: Ada kabar olahraga Indonesia yang membanggakan?
(opsi: {'sentiment': 'positif'})

Kategori query terdeteksi : sport (0.97)
Filter dipakai            : {'kategori': 'sport', 'sentiment': 'positif'}
Ringkasan sentimen sumber : {'positif': 5}

JAWABAN:
Ya, ada beberapa kabar olahraga Indonesia yang membanggakan:

*   Indonesia menjadi tuan rumah kompetisi grappling paling bergengsi di dunia, ADCC, yang menunjukkan posisi strategis Indonesia di Asia Tenggara dan pertumbuhan pasar grappling yang cepat [3].
*   Atlet arung jeram Indonesia meraih medali emas, satu perak, dan satu perunggu pada IRF World Rafting Championship 2025 di Malaysia [4].
*   KORMI meluncurkan Pedoman Nasional Gerakan Indonesia Aktif untuk mengoptimalkan potensi olahraga masyarakat demi mewujudkan SDM unggul dan berdaya saing, serta mendukung sport tourism dan menggerakkan ekonomi [1].
*   Pasangan Fajar Alfian/Muhammad Shihibul Fikri berhasil merebut tahta tertinggi di turnamen Super 1000 China Open pada J

Pengujian ketiga sekaligus menguji penyaringan sentimen, dengan pertanyaan "Ada kabar olahraga Indonesia yang membanggakan?" disertai permintaan sentimen positif. Pertanyaan diklasifikasikan sebagai sport dengan keyakinan sangat tinggi (0,97), lalu filter gabungan kategori sport dan sentimen positif diterapkan. Seluruh lima sumber yang diambil merupakan berita olahraga bersentimen positif dengan jarak yang kecil, dan jawaban merangkum berbagai kabar membanggakan seperti penyelenggaraan ADCC, medali arung jeram, dan prestasi bulu tangkis, lengkap dengan sitasi. Pengujian ini membuktikan integrasi penuh ketiga komponen, di mana klasifikasi, penyaringan sentimen, dan retrieval bekerja bersama untuk menjawab pertanyaan yang tidak mungkin dijawab tanpa ketiganya.

In [14]:
# Pertanyaan agregasi sentimen
tampilkan("Bagaimana kabar dan sentimen berita kesehatan akhir-akhir ini?")

PERTANYAAN: Bagaimana kabar dan sentimen berita kesehatan akhir-akhir ini?

Kategori query terdeteksi : health (0.97)
Filter dipakai            : {'kategori': 'health'}
Ringkasan sentimen sumber : {'negatif': 4, 'netral': 1}

JAWABAN:
Kabar berita kesehatan akhir-akhir ini menunjukkan sentimen yang cenderung negatif, terutama terkait peningkatan kasus penyakit serius dan kronis, serta masalah kesadaran kesehatan masyarakat.

Beberapa poin utama:
*   **Peningkatan Penyakit Jantung dan Kronis pada Usia Muda:** Banyak anak muda di Malaysia mengalami kerusakan jantung dan penyakit kronis lainnya seperti hipertensi, yang dikaitkan dengan gaya hidup modern, stres, kurang tidur, minim olahraga, merokok, dan konsumsi makanan olahan [1, 5]. Serangan jantung juga ditekankan sebagai kondisi darurat yang dapat menyebabkan kerusakan permanen pada otot jantung [3].
*   **Rendahnya Kesadaran dan Penundaan Pengobatan:** Masyarakat, khususnya di Malaysia, sering menunda berobat karena biaya, pekerjaan,

Pengujian keempat menyoroti pemanfaatan agregasi sentimen, dengan pertanyaan "Bagaimana kabar dan sentimen berita kesehatan akhir-akhir ini?". Pertanyaan diklasifikasikan sebagai health (0,97), dan ringkasan sentimen dari sumber yang diambil menunjukkan dominasi negatif (4 negatif, 1 netral). Informasi agregat ini tercermin langsung dalam jawaban, yang menyimpulkan sentimen berita kesehatan cenderung negatif lalu merinci poin-poin utamanya seperti peningkatan penyakit kronis dan penundaan pengobatan, sembari tetap mencantumkan sisi positif berupa kisah kesembuhan. Pengujian ini menunjukkan agregasi sentimen tidak sekadar menjadi metadata pasif, melainkan turut membentuk substansi jawaban, sehingga sistem mampu menjawab pertanyaan yang menuntut gambaran kecenderungan opini secara menyeluruh.

In [12]:
# Tes anti-ngarang: pertanyaan yang TIDAK ada di dataset
tampilkan("Berapa harga tiket konser Coldplay di Jakarta?")

PERTANYAAN: Berapa harga tiket konser Coldplay di Jakarta?

Kategori query terdeteksi : sport (0.42)
Filter dipakai            : {}
Ringkasan sentimen sumber : {'positif': 3, 'netral': 1, 'negatif': 1}

JAWABAN:
Saya tidak menemukan informasi mengenai harga tiket konser Coldplay di Jakarta dalam artikel yang tersedia.

SUMBER:
  [1] (sport/positif, dist=0.358) Jadwal SBY Cup 2026 di Magelang: Tim Peserta, Harga Tiket, d
      https://sport.detik.com/sport-lain/d-8484348/jadwal-sby-cup-2026-di-magelang-tim-peserta-harga-tiket-dan-siaran-tv
  [2] (finance/netral, dist=0.449) Catat Tanggalnya! Diskon Sampai 70% Bakal Banjiri Mal-mal Ja
      https://finance.detik.com/berita-ekonomi-bisnis/d-8497082/catat-tanggalnya-diskon-sampai-70-bakal-banjiri-mal-mal-jakarta
  [3] (sport/positif, dist=0.478) Indonesia Jadi Tuan Rumah Panggung Grappling Dunia!
      https://sport.detik.com/sport-lain/d-8497309/indonesia-jadi-tuan-rumah-panggung-grappling-dunia
  [4] (sport/positif, dist=0.521) SKORZ FX 

Pengujian terakhir menguji kemampuan anti-halusinasi dengan pertanyaan yang sengaja berada di luar cakupan data, yaitu "Berapa harga tiket konser Coldplay di Jakarta?". Karena topik ini tidak benar-benar tercakup kategori manapun, classifier menghasilkan keyakinan rendah (sport, 0,42) sehingga sistem tidak memaksakan filter kategori. Meskipun retrieval tetap mengembalikan beberapa chunk yang secara longgar berkaitan dengan kata "tiket", LLM tidak mengarang jawaban dan secara jujur menyatakan bahwa informasi tersebut tidak ditemukan dalam artikel yang tersedia.

Pengujian ini merupakan bukti penting bahwa mekanisme grounding bekerja sebagaimana mestinya: alih-alih menghasilkan jawaban palsu yang terkesan meyakinkan, sistem mengakui keterbatasan datanya. Inilah perilaku yang diharapkan dari sistem RAG yang tepercaya, sekaligus menegaskan keunggulan pendekatan ini dibandingkan LLM yang dipakai tanpa grounding.

## Penutup

Tahap ini telah berhasil merangkai seluruh komponen menjadi satu pipeline RAG yang utuh dan berfungsi. Melalui serangkaian pengujian, sistem terbukti mampu menjawab beragam tipe pertanyaan: pertanyaan spesifik yang menuntut sintesis fakta, pertanyaan ringkasan atas suatu kategori, pertanyaan dengan penyaringan sentimen, pertanyaan yang menuntut gambaran agregat sentimen, hingga pertanyaan di luar cakupan data yang dijawab secara jujur tanpa mengarang. Seluruh jawaban dilengkapi sitasi sumber yang dapat ditelusuri.

Pengujian ini sekaligus menegaskan keberhasilan integrasi tiga komponen utama: klasifikasi berperan dalam pengarahan pencarian (routing), analisis sentimen berperan dalam penyaringan dan agregasi, sedangkan retrieval dan generation menjadi inti tanya-jawab. Dengan pipeline yang telah terbukti bekerja, sistem siap dibungkus ke dalam antarmuka aplikasi agar dapat digunakan secara interaktif oleh pengguna akhir.